# PLM_Agent — NEHM Global Dictionary → Atom Labels → Sample Description

This notebook uses the self-contained local `PLM_Agent` package with NEHM Pipeline B embeddings (`student_z_2304_B.npy`). It has no runtime dependency on VisualHistory or Azure.

**Stages**
1. Learn / load a **cached global dictionary** (approx K-SVD, **400 atoms**) with **non-negative Elastic Net** (`α=1e-4`, `l1_ratio=0.5`).
2. Retrieve **top-40 corpus hits per atom** and run **local Qwen RCS** (`qwen2.5:7b`) over Material Database metadata (**all fields except logit columns**).
3. Load the stable, reusable **per-atom microscopy labels**; the established label cache is independent of the sample coordinator toggle.
4. Describe an **arbitrary sample** with the selected coordinator, led by **FAISS L2** (type/identity) and **vision** (image appearance), with **dictionary atoms as critical evidence that must be interpreted** (not copied as identity; positive evidence only after type is established).

Set `COORDINATOR_CHOICE` in the setup cell to `"gpt-5.1"` or `"qwen3.8:27b"`.

Artifacts land under `NEHM_RESULTS/student_inference_B/global_dictionary_interpretability/`.

**Configuration:** private credentials live in `PLM_Agent/.env`; local model and safe defaults are documented in `PLM_Agent/.env.example`.


In [ ]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _detect_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "nehm_pipeline").is_dir() and (cand / "Database").is_dir():
            return cand
    raise FileNotFoundError("Could not auto-detect PROJECT_ROOT")


PROJECT_ROOT = _detect_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# The package may have been created after this Jupyter kernel started.
importlib.invalidate_caches()
import PLM_Agent.atom_interpretability as ai
importlib.reload(ai)

MATERIAL_DB_XLSX = PROJECT_ROOT / "Database" / "Material_Database.xlsx"
IMAGES_DIR = PROJECT_ROOT / "NEHM"
Z_PATH = PROJECT_ROOT / "NEHM_RESULTS" / "student_inference_B" / "student_z_2304_B.npy"
OUT_DIR = PROJECT_ROOT / "NEHM_RESULTS" / "student_inference_B" / "global_dictionary_interpretability"
PATHS = ai.cache_paths(OUT_DIR)

# Self-contained PLM_Agent configuration (direct OpenAI + local Ollama; no Azure).
PLM_ENV_PATH = ai.load_env()

# Tunables
FORCE_RETRAIN_DICT = False          # True to rebuild dictionary even if cache exists
FORCE_REBUILD_RETRIEVAL = False     # True to recompute top-40 hits
FORCE_RELABEL_ATOMS = False         # True only to intentionally rebuild the stable label vocabulary
RCS_MAX_ATOMS = None                # e.g. 5 for smoke test; None = all 400
LABEL_MAX_ATOMS = None              # e.g. 5 for smoke test; None = all labeled atoms with RCS

# Per-atom labels define a stable dictionary vocabulary and remain on their existing cache.
# This model is used only to fill genuinely missing labels or when FORCE_RELABEL_ATOMS=True.
ATOM_LABEL_BACKEND = "openai"
ATOM_LABEL_MODEL = "gpt-5.1"

# Coordinator toggle — controls sample reranking, synthesis, and vision only.
COORDINATOR_CHOICE = "gpt-5.1"      # "gpt-5.1" | "qwen3.8:27b"
if COORDINATOR_CHOICE == "gpt-5.1":
    COORDINATOR_BACKEND = "openai"
    COORDINATOR_MODEL = "gpt-5.1"
elif COORDINATOR_CHOICE == "qwen3.8:27b":
    COORDINATOR_BACKEND = "ollama"
    COORDINATOR_MODEL = "qwen3.8:27b"
else:
    raise ValueError(f"Unsupported COORDINATOR_CHOICE={COORDINATOR_CHOICE!r}")

QUERY_ROW = 4505                    # change freely

# Dictionary-learning tunables (Visual History Run4 defaults, n_atoms=400)
KSVD_N_ATOMS = 400                  # number of dictionary atoms
KSVD_MAX_ITER = 10                  # outer approx K-SVD iterations
KSVD_SUBSAMPLE_FRACTION = 0.1       # fraction of samples per outer iter (1.0 = full batch)
KSVD_ELASTIC_NET_ALPHA = 1e-4       # Elastic Net α
KSVD_ELASTIC_NET_L1_RATIO = 0.5     # Elastic Net l1_ratio
KSVD_ELASTIC_NET_MAX_ITER = 1000    # inner Elastic Net solver iterations
KSVD_NONNEGATIVE = True             # non-negative sparse codes
KSVD_ACTIVE_EPS = 1e-4
KSVD_SEED = 42

KSVD_CFG = ai.ClassicalKsvdConfig(**{
    **{f.name: getattr(ai.NEHM_KSVD_CONFIG, f.name)
       for f in ai.ClassicalKsvdConfig.__dataclass_fields__.values()},
    "n_atoms": int(KSVD_N_ATOMS),
    "max_iter": int(KSVD_MAX_ITER),
    "subsample_fraction": float(KSVD_SUBSAMPLE_FRACTION),
    "elastic_net_alpha": float(KSVD_ELASTIC_NET_ALPHA),
    "elastic_net_l1_ratio": float(KSVD_ELASTIC_NET_L1_RATIO),
    "elastic_net_max_iter": int(KSVD_ELASTIC_NET_MAX_ITER),
    "sparse_codes_nonnegative": bool(KSVD_NONNEGATIVE),
    "active_eps": float(KSVD_ACTIVE_EPS),
    "seed": int(KSVD_SEED),
})

print("PROJECT_ROOT      :", PROJECT_ROOT)
print("Z_PATH            :", Z_PATH, "exists=", Z_PATH.is_file())
print("OUT_DIR           :", OUT_DIR)
print("PLM_AGENT_ENV     :", PLM_ENV_PATH)
print("ATOM LABEL MODEL  :", ATOM_LABEL_BACKEND, "/", ATOM_LABEL_MODEL, "(stable cache)")
print("SAMPLE COORDINATOR:", COORDINATOR_BACKEND, "/", COORDINATOR_MODEL)
print(
    "KSVD config       :",
    f"{KSVD_CFG.n_atoms} atoms | max_iter={KSVD_CFG.max_iter} | "
    f"subsample={KSVD_CFG.subsample_fraction} | "
    f"elastic_net α={KSVD_CFG.elastic_net_alpha} "
    f"l1_ratio={KSVD_CFG.elastic_net_l1_ratio} "
    f"nonneg={KSVD_CFG.sparse_codes_nonnegative}"
)

## 1. Load aligned corpus

Student embeddings are `(5410, 2304)`. The spreadsheet may be longer; we align to the embedding row count and keep every Material Database field **except logit columns** (`logitb`, `Run*_Logit`).

In [ ]:
Z, DF, META_FIELDS = ai.load_aligned_corpus(
    z_path=Z_PATH,
    xlsx_path=MATERIAL_DB_XLSX,
    sheet_name="Sheet2",
    images_dir=IMAGES_DIR,
)
print(f"Z={Z.shape}  DF={DF.shape}")
print("Metadata fields used for RCS (logit fields excluded):")
for c in META_FIELDS:
    print(" -", c)
print("Excluded logit-like fields:", [c for c in DF.columns if ai.is_logit_field(c)])

## 2. Train / load global dictionary (approx K-SVD + non-negative Elastic Net)

Uses the same approximate K-SVD + Elastic Net implementation as Visual History / Painting Interpretability (`classical_ksvd.py`), with:
- `n_atoms = 400`
- `elastic_net_alpha = 1e-4`
- `elastic_net_l1_ratio = 0.5`
- `sparse_codes_nonnegative = True`
- mean-centered embeddings (`preprocess_mode='raw'` + explicit mean subtraction)

Tune `KSVD_MAX_ITER`, `KSVD_N_ATOMS`, etc. in the setup cell. Set `FORCE_RETRAIN_DICT = True` to rebuild after changing them.

In [ ]:
dict_bundle = ai.train_or_load_dictionary(
    Z,
    OUT_DIR,
    cfg=KSVD_CFG,
    force_retrain=FORCE_RETRAIN_DICT,
)
D = dict_bundle["D"]
MEAN = dict_bundle["mean"]
CODES = dict_bundle["codes"]
CFG = dict_bundle["cfg"]
print(f"D={D.shape}  MEAN={MEAN.shape}  CODES={CODES.shape}  cached={dict_bundle['cached']}")
print(f"mean active atoms/sample (eps={CFG.active_eps}):",
      float((CODES > CFG.active_eps).sum(axis=1).mean()))
display(pd.read_csv(PATHS["usage"]).head(15) if PATHS["usage"].is_file() else "(usage csv not yet written)")

## 2b. Inspect an atom (quality check)

Plot one dictionary atom across the 2304-D teacher blocks (`image 768 | text×3 512`), plus its top triggered specimens and reconstruction contribution. Change `PLOT_ATOM_ID` (or leave as `"auto"` to pick the most-used atom).

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

PLOT_ATOM_ID = "auto"   # int atom id, or "auto" = highest-usage atom
PLOT_TOP_K = 6          # exemplar thumbnails
BLOCK_DELIMS = [768, 1280, 1792]
BLOCK_LABELS = ["image\n768", "text1\n512", "text2\n512", "text3\n512"]

usage = (CODES > float(CFG.active_eps)).sum(axis=0)
if PLOT_ATOM_ID == "auto":
    atom_id = int(np.argmax(usage))
else:
    atom_id = int(PLOT_ATOM_ID) % int(D.shape[0])

atom = np.asarray(D[atom_id], dtype=np.float32)
codes_a = np.asarray(CODES[:, atom_id], dtype=np.float32)
active_rows = np.flatnonzero(codes_a > float(CFG.active_eps))
top_rows = active_rows[np.argsort(codes_a[active_rows])[::-1][:PLOT_TOP_K]] if active_rows.size else np.array([], dtype=int)

# Energy per teacher block
edges = [0] + BLOCK_DELIMS + [atom.size]
block_energy = [float(np.linalg.norm(atom[a:b]) ** 2) for a, b in zip(edges[:-1], edges[1:])]
block_energy = np.asarray(block_energy)
block_energy = block_energy / max(float(block_energy.sum()), 1e-12)

fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(2, 2, height_ratios=[1.35, 1.0], width_ratios=[2.2, 1.0])

# --- atom waveform ---
ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(atom, lw=0.9, color="#1f4e79")
for x in BLOCK_DELIMS:
    ax0.axvline(x, color="black", lw=0.8, alpha=0.45)
ax0.set_title(
    f"Atom {atom_id}  |  usage={(usage[atom_id]/len(CODES)):.1%} of corpus  |  "
    f"max code={float(codes_a.max()):.4f}  |  ‖atom‖={float(np.linalg.norm(atom)):.3f}"
)
ax0.set_xlabel("dimension in mean-centered 2304-D teacher space")
ax0.set_ylabel("atom weight")
ax0.grid(alpha=0.25)
mids = [(edges[i] + edges[i + 1]) / 2 for i in range(4)]
ymin, ymax = ax0.get_ylim()
for mid, lab in zip(mids, BLOCK_LABELS):
    ax0.text(mid, ymax - 0.04 * (ymax - ymin), lab, ha="center", va="top", fontsize=9, color="#333")

# --- block energy ---
ax1 = fig.add_subplot(gs[0, 1])
ax1.bar(["image", "text1", "text2", "text3"], block_energy, color=["#4c78a8", "#f58518", "#e45756", "#72b7b2"])
ax1.set_ylim(0, 1)
ax1.set_ylabel("fraction of ‖atom‖²")
ax1.set_title("Block energy")
for i, v in enumerate(block_energy):
    ax1.text(i, v + 0.02, f"{v:.0%}", ha="center", fontsize=9)

# --- top exemplars ---
ax_row = fig.add_subplot(gs[1, :])
ax_row.axis("off")
ax_row.set_title(f"Top-{len(top_rows)} specimens triggering atom {atom_id}", pad=8)
if top_rows.size == 0:
    ax_row.text(0.5, 0.5, "no active specimens", ha="center", va="center")
else:
    thumbs = []
    labels = []
    for ridx in top_rows:
        img_name = str(DF.iloc[int(ridx)].get("Image", "") or "")
        spec = str(DF.iloc[int(ridx)].get("Specimen Name", "") or "")[:28]
        src = str(DF.iloc[int(ridx)].get("Source", "") or "")[:14]
        path = IMAGES_DIR / img_name
        if path.is_file():
            im = Image.open(path).convert("RGB")
            im.thumbnail((180, 180))
            thumbs.append(np.asarray(im))
        else:
            thumbs.append(np.full((120, 120, 3), 230, dtype=np.uint8))
        labels.append(f"row {int(ridx)}\ncode={codes_a[int(ridx)]:.3f}\n{spec}\n{src}")

    n = len(thumbs)
    inner = ax_row.inset_axes([0.0, 0.05, 1.0, 0.9])
    inner.axis("off")
    for i, (im, lab) in enumerate(zip(thumbs, labels)):
        x0 = i / n
        ax_i = inner.inset_axes([x0 + 0.01, 0.22, 1 / n - 0.02, 0.78])
        ax_i.imshow(im)
        ax_i.axis("off")
        inner.text(x0 + 0.5 / n, 0.08, lab, ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

print("Block energy:", {k: float(v) for k, v in zip(["image", "text1", "text2", "text3"], block_energy)})
print("Active specimens:", int(active_rows.size), "/", len(CODES))
if PATHS["history"].is_file():
    hist = pd.read_csv(PATHS["history"])
    print("Last K-SVD iter MSE:", float(hist["mse"].iloc[-1]) if "mse" in hist.columns else hist.tail(1).to_dict())


## 3. Top-40 atom retrieval + local Qwen RCS

For each atom:
1. Cosine-retrieve the top 40 mean-centered embedding neighbors over the whole corpus
2. Build a metadata passage from **all non-logit fields**
3. Chunk with PLM_Agent's internal overlapping atomic chunker
4. Score/summarize chunks with **local Ollama `qwen2.5:7b`** (RCS)

Tips for first run: set `RCS_MAX_ATOMS = 5` in setup for a smoke test. Full 400-atom RCS is slow but fully cached in `qwen_rcs_chunk_cache.jsonl`.

In [ ]:
if PATHS["retrieval"].is_file() and not FORCE_REBUILD_RETRIEVAL:
    retrieval = ai.load_retrieval_jsonl(PATHS["retrieval"])
    print(f"Loaded cached retrieval: {len(retrieval)} atoms from {PATHS['retrieval']}")
else:
    retrieval = ai.retrieve_top_hits_for_atoms(
        D, Z, MEAN, DF, META_FIELDS, top_k=ai.TOP_K_DEFAULT
    )
    ai.save_retrieval_jsonl(PATHS["retrieval"], retrieval)
    print(f"Wrote retrieval → {PATHS['retrieval']}")

# Peek one atom
ex = retrieval[0]
print(f"Atom {ex['atom_id']} top hit: row={ex['hits'][0]['database_row']} "
      f"specimen={ex['hits'][0]['specimen']!r} cos={ex['hits'][0]['cosine']:.4f}")
print("Chunk count (hit 0):", len(ex["hits"][0].get("chunks", [])))

rcs_cache = ai.run_qwen_rcs_for_retrieval(
    retrieval,
    PATHS["rcs_cache"],
    max_atoms=RCS_MAX_ATOMS,
)
print(f"RCS cache size: {len(rcs_cache)} chunks → {PATHS['rcs_cache']}")

## 4. Stable per-atom labels

Per-atom labels define the dictionary's persistent vocabulary. Changing `COORDINATOR_CHOICE` for sample descriptions does **not** regenerate or replace them. Existing records in `atom_microscopy_labels.jsonl` are always reused.

Only genuinely missing labels are generated with `ATOM_LABEL_MODEL` (GPT-5.1 by default). Set `FORCE_RELABEL_ATOMS=True` only when intentionally rebuilding the atom vocabulary.

Outputs:
- `atom_microscopy_labels.json` / `.jsonl` under the cache directory

Use `LABEL_MAX_ATOMS = 5` for a smoke test before labeling all atoms.

In [ ]:
import importlib
importlib.reload(ai)

print(
    f"Starting atom labels | LABEL_MAX_ATOMS={LABEL_MAX_ATOMS} "
    f"FORCE_RELABEL={FORCE_RELABEL_ATOMS} → {PATHS['atom_labels_jsonl']}",
    flush=True,
)
atom_labels = ai.synthesize_atom_labels(
    retrieval,
    rcs_cache,
    PATHS["atom_labels"],
    PATHS["atom_labels_jsonl"],
    coordinator_backend=ATOM_LABEL_BACKEND,
    coordinator_model=ATOM_LABEL_MODEL,
    max_atoms=LABEL_MAX_ATOMS,
    force=FORCE_RELABEL_ATOMS,
)
LABELS = {int(r["atom_id"]): r for r in atom_labels}
print(f"Labeled atoms: {len(LABELS)}")
for atom_id in sorted(LABELS)[:10]:
    rec = LABELS[atom_id]
    print(f"  [{atom_id:03d}] {rec.get('confidence','?'):6s}  {rec.get('label','')}")


## 5. Describe an arbitrary sample

For `QUERY_ROW`, roles of evidence:

1. **Nearest neighbors** — lead for likely type / identity / catalog context. The most frequently repeated specimen is prioritized (distance breaks count ties); every neighbor's chemical formula is shown, and a formula recurring at least three times is suggested in the description
2. **Direct vision** — lead for what is visible in the query image and estimate magnification
3. **Dictionary atoms** — **critical**, but **interpret**: use transferable attributes (color, interference, illumination, relief, texture, surface pitting, habit). Example: atom "Pitted softwood tracheid pulp fibers" on acrylic may mean delustrant pitting, not softwood pulp. Keep established identity; do not paste atom labels as identity

Description requirements:
- Name the recorded **illumination modality in sentence 1**
- Give an **estimated magnification in sentence 1 or 2**
- Treat atoms contributing at least 1% as **candidate evidence** and use only those that add relevant, compatible attributes
- Always use the **top-ranked atom title** in the description's language and cite that atom exactly once
- Cite every other selected atom ID **at most once** across the complete description
- After type is established, report **positive evidence only**
- If coordinator synthesis exhausts its retries, save a deterministic **L2 + vision fallback** instead of failing the cell

Change `QUERY_ROW` in the setup cell and re-run this section.


In [ ]:
import importlib
importlib.reload(ai)

from IPython.display import Markdown, display
from PIL import Image
import matplotlib.pyplot as plt

sample_out = PATHS["sample_dir"] / f"row_{QUERY_ROW:05d}_description.json"
result = ai.describe_sample(
    query_row=3924, #QUERY_ROW,
    z=Z,
    df=DF,
    codes=CODES,
    labels=LABELS,
    images_dir=IMAGES_DIR,
    out_path=sample_out,
    cfg=CFG,
    coordinator_backend=COORDINATOR_BACKEND,
    coordinator_model=COORDINATOR_MODEL,
    faiss_k=10,
)

q = int(result["target"]["database_row"])
img_path = IMAGES_DIR / result["target"]["image"]
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(Image.open(img_path).convert("RGB"))
ax.set_title(f"row {q} | {result['target'].get('specimen','')} | {result['target'].get('source','')}")
ax.axis("off")
plt.show()

print("Evidence priority:", result.get("pipeline", {}).get("evidence_priority"))
print("Synthesis fallback used:", result.get("pipeline", {}).get("synthesis_fallback_used", False))
print("Established type:", result.get("established_type"))
print("Illumination:", result.get("target", {}).get("illumination_modality"))
print("Magnification estimate:", result.get("magnification_estimate"))
print(
    "Candidate atoms (>=1%):",
    result.get("pipeline", {}).get("included_atom_count"),
    "/ active:",
    result.get("pipeline", {}).get("active_atom_count"),
)

print("\nNearest neighbors (primary):")
for h in result["faiss_neighbors"]:
    print(
        f"  row={h['database_row']:5d} d2={h['d2']:.3f}  "
        f"{h['specimen'][:40]:40s}  formula={h.get('chemical_formula', '')}  {h['source']}"
    )

consensus = result.get("faiss_l2_consensus", {})
print("\nMost repeated L2 specimen:", consensus.get("most_repeated_specimen"))
print("Repeated formula suggestion:", consensus.get("repeated_formula_suggestion"))

vision = result.get("direct_visual_observations", {})
print("\nVision morphology (secondary):", vision.get("morphology"))

print("\nCandidate atoms (>=1%; only relevant atoms are cited, at most once):")
for a in result["weighted_atoms"]:
    print(f"  atom {a['atom_id']:3d}  {a['importance_percent']:5.1f}%  {a['label']}")

display(Markdown("### Microscopy description\n" + str(result.get("microscopy_description", ""))))
print("Wrote:", sample_out)


## Notes

- `PLM_Agent` is self-contained; no VisualHistory or Azure runtime is used.
- Dictionary / codes / RCS / labels are cached; re-running is incremental and model-aware.
- Local Ollama requires `qwen2.5:7b` for RCS and `qwen3.8:27b` when selected as coordinator.
- Direct GPT-5.1 uses `OPENAI_API_KEY` from the private `PLM_Agent/.env`.
- If atom labeling is interrupted, resume from `atom_microscopy_labels.jsonl`; cached labels are reused independently of the sample coordinator unless `FORCE_RELABEL_ATOMS=True`.
- Sample descriptions: FAISS L2 + vision lead type/appearance; atoms are critical interpretive evidence; written under `sample_descriptions/row_XXXXX_description.json`.
